In [66]:
print(123)

123


In [67]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [68]:
# The first time you run this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. 
# The tokenizer turns text into something the model can read. After that, both load from a local cache.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [69]:
# how embeddings works
# v1 is a vector, an array of 384 numbers. 
# Each number stands for some concept the model learned. 
# We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.
v1 = model.encode(q1) 

In [70]:
v1

array([-7.94797949e-03, -9.18932408e-02, -1.14074284e-02,  2.18466595e-02,
        1.11858333e-02, -1.30818449e-02, -7.39962235e-02, -9.87466797e-02,
       -1.05911419e-01, -3.03381458e-02, -2.92174686e-02,  2.33883355e-02,
        7.87481945e-03,  4.15800288e-02,  2.42720488e-02, -3.65723185e-02,
       -5.31095453e-02, -1.94869116e-02, -2.26979833e-02,  8.76777805e-03,
       -1.10785306e-01,  3.97906862e-02, -4.18479182e-02,  2.82960795e-02,
       -1.28302518e-02,  1.72567349e-02,  3.10428441e-02,  9.51102898e-02,
       -1.90717615e-02, -6.23236150e-02, -3.59101109e-02,  6.46180734e-02,
        3.06465179e-02,  2.39972025e-02,  2.06126980e-02,  9.87384096e-03,
        7.63835683e-02, -6.50510937e-02,  4.07937029e-03,  2.34527756e-02,
       -2.49407142e-02, -2.95020584e-02, -1.74879059e-02,  4.62139063e-02,
        2.48923320e-02,  1.08981736e-01, -5.67837544e-02, -6.95324540e-02,
        4.35405597e-03,  2.85132304e-02,  2.75795385e-02, -2.49844193e-02,
       -6.82148617e-03,  

In [71]:
v1.shape

(384,)

In [72]:
# encode documents 
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [73]:
# we compare the query against the document using dot product
v1.dot(dv)

np.float32(0.39572883)

In [74]:
# Now we try an unrelated query
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [75]:
v2.dot(dv)

np.float32(0.019730574)

In [76]:
# The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. 
# The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. 
# A score near 0 means the two vectors are about as different as they can be.
# That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

In [77]:
# we created ingest.py for loading the FAQ data. Download it into your project:

In [78]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-08-19 22:24:27--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.5’

ingest.py.5         100%[===================>]     738  --.-KB/s    in 0s      

2026-08-19 22:24:27 (21.9 MB/s) - ‘ingest.py.5’ saved [738/738]



In [79]:
from ingest import load_faq_data

documents = load_faq_data()

# Generating embeddings

In [80]:
# Each document is a Python dictionary with a question and an answer. 
# We embed both together. That way a query can match against the question text and the answer text in our index.
# Build one text per document:

In [81]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [82]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [83]:
len(texts)

1406

In [84]:
# Now we generate the embeddings. 
# We have about 1200 texts here. We won't hand the model all of them at once. 
# That takes a long time, and we can't see what's happening inside. Instead we split them into batches.
# First we import tqdm so we can watch the progress

In [85]:
from tqdm.auto import tqdm

In [86]:
# Next we chunk the dataset into batches of 50 and encode each batch:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)


  0%|          | 0/29 [00:00<?, ?it/s]

1406

In [87]:
# We end up with 1406 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.
# We turn them into a 2-dimensional array (matrix) where rows are documents (vectors) and columns are dimensions of the vectors

In [88]:
import numpy as np
X = np.array(vectors)

In [89]:
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [ 0.00878648, -0.07507781,  0.02732081, ..., -0.00520807,
         0.01720901,  0.03526433],
       [-0.01129575,  0.04223463, -0.03605105, ..., -0.03297513,
        -0.00711083, -0.01410813],
       [-0.0185939 , -0.00951876, -0.05152111, ...,  0.04781627,
         0.0265054 ,  0.07470339]], shape=(1406, 384), dtype=float32)

In [90]:
X.shape

(1406, 384)

In [91]:
scores = X.dot(v1) 
# This is matrix-vector multiplication.


In [92]:
scores

array([0.41303527, 0.26119864, 0.60880876, ..., 0.23222184, 0.20574081,
       0.146424  ], shape=(1406,), dtype=float32)

In [93]:
# The highest score is the most similar document:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(1009), np.float32(0.8317791))

In [94]:
documents[idx]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [95]:
# Usually we want more than the single best match, so let's pull the top 5.
# np.argsort sorts from lowest to highest, so the last 5 are the top ones:

In [96]:
np.argsort(scores)

array([ 750,  217, 1044, ..., 1155,  567, 1009], shape=(1406,))

In [97]:
top5 = np.argsort(scores)[-5:]

In [98]:
top5

array([ 503,    2, 1155,  567, 1009])

In [99]:
# They come out smallest-first, so we reverse them to get the highest first:

In [100]:
top5 = top5[::-1]
top5

array([1009,  567, 1155,    2,  503])

In [101]:
scores[top5]

array([0.8317791 , 0.6845695 , 0.61755615, 0.60880876, 0.58479667],
      dtype=float32)

In [102]:
top5 = np.argsort(-scores)[:5]

In [103]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.8317791
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6845695
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.61755615
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 's

In [104]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [105]:
vindex.search(v1)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [106]:
vindex.search(v1, num_results=5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [107]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [108]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-08-19 22:25:36--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.3’

rag_helper.py.3     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-08-19 22:25:36 (18.4 MB/s) - ‘rag_helper.py.3’ saved [2134/2134]



In [109]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [110]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [111]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [112]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [113]:
# This still uses keyword search. Text search isn't bad here, so the answer may already look right. Next we replace search with vector search.

In [114]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [115]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [116]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [119]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [121]:
vs_index.clear()
vs_index.fit(vectors, documents)

In [122]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [123]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [124]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [125]:
vs_index.close()